In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.preprocessing import LabelEncoder,StandardScaler
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

In [2]:
data = pd.read_csv('ai_job_dataset.csv')
data.head()

,job_id,job_title,salary_usd,salary_currency,experience_level,employment_type,company_location,company_size,employee_residence,remote_ratio,required_skills,education_required,years_experience,industry,posting_date,application_deadline,job_description_length,benefits_score,company_name
0,AI00001,AI Research Scientist,90376,USD,SE,CT,China,M,China,50,"Tableau, PyTorch, Kubernetes, Linux, NLP",Bachelor,9,Automotive,2024-10-18,2024-11-07,1076,5.9,Smart Analytics
1,AI00002,AI Software Engineer,61895,USD,EN,CT,Canada,M,Ireland,100,"Deep Learning, AWS, Mathematics, Python, Docker",Master,1,Media,2024-11-20,2025-01-11,1268,5.2,TechCorp Inc
2,AI00003,AI Specialist,152626,USD,MI,FL,Switzerland,L,South Korea,0,"Kubernetes, Deep Learning, Java, Hadoop, NLP",Associate,2,Education,2025-03-18,2025-04-07,1974,9.4,Autonomous Tech
3,AI00004,NLP Engineer,80215,USD,SE,FL,India,M,India,50,"Scala, SQL, Linux, Python",PhD,7,Consulting,2024-12-23,2025-02-24,1345,8.6,Future Systems
4,AI00005,AI Consultant,54624,EUR,EN,PT,France,S,Singapore,100,"MLOps, Java, Tableau, Python",Master,0,Media,2025-04-15,2025-06-23,1989,6.6,Advanced Robotics


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   job_id                  15000 non-null  object 
 1   job_title               15000 non-null  object 
 2   salary_usd              15000 non-null  int64  
 3   salary_currency         15000 non-null  object 
 4   experience_level        15000 non-null  object 
 5   employment_type         15000 non-null  object 
 6   company_location        15000 non-null  object 
 7   company_size            15000 non-null  object 
 8   employee_residence      15000 non-null  object 
 9   remote_ratio            15000 non-null  int64  
 10  required_skills         15000 non-null  object 
 11  education_required      15000 non-null  object 
 12  years_experience        15000 non-null  int64  
 13  industry                15000 non-null  object 
 14  posting_date            15000 non-null

In [4]:
def change_to_date(column):
    data[column] = pd.to_datetime(data[column],infer_datetime_format=True)

In [5]:
data.columns

Index(['job_id', 'job_title', 'salary_usd', 'salary_currency',
       'experience_level', 'employment_type', 'company_location',
       'company_size', 'employee_residence', 'remote_ratio', 'required_skills',
       'education_required', 'years_experience', 'industry', 'posting_date',
       'application_deadline', 'job_description_length', 'benefits_score',
       'company_name'],
      dtype='object')

In [6]:
for i in ['posting_date','application_deadline']:
    change_to_date(i)

In [7]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   job_id                  15000 non-null  object        
 1   job_title               15000 non-null  object        
 2   salary_usd              15000 non-null  int64         
 3   salary_currency         15000 non-null  object        
 4   experience_level        15000 non-null  object        
 5   employment_type         15000 non-null  object        
 6   company_location        15000 non-null  object        
 7   company_size            15000 non-null  object        
 8   employee_residence      15000 non-null  object        
 9   remote_ratio            15000 non-null  int64         
 10  required_skills         15000 non-null  object        
 11  education_required      15000 non-null  object        
 12  years_experience        15000 non-null  int64 

In [8]:
data['posting_Date'] = data['posting_date'].dt.day
data['posting_month'] = data['posting_date'].dt.month
data['posting_year'] = data['posting_date'].dt.year

In [9]:
data['application_date'] = data['application_deadline'].dt.day
data['application_month'] = data['application_deadline'].dt.month
data['application_year'] = data['application_deadline'].dt.year

In [10]:
data.head()

,job_id,job_title,salary_usd,salary_currency,experience_level,employment_type,company_location,company_size,employee_residence,remote_ratio,...,application_deadline,job_description_length,benefits_score,company_name,posting_Date,posting_month,posting_year,application_date,application_month,application_year
0,AI00001,AI Research Scientist,90376,USD,SE,CT,China,M,China,50,...,2024-11-07,1076,5.9,Smart Analytics,18,10,2024,7,11,2024
1,AI00002,AI Software Engineer,61895,USD,EN,CT,Canada,M,Ireland,100,...,2025-01-11,1268,5.2,TechCorp Inc,20,11,2024,11,1,2025
2,AI00003,AI Specialist,152626,USD,MI,FL,Switzerland,L,South Korea,0,...,2025-04-07,1974,9.4,Autonomous Tech,18,3,2025,7,4,2025
3,AI00004,NLP Engineer,80215,USD,SE,FL,India,M,India,50,...,2025-02-24,1345,8.6,Future Systems,23,12,2024,24,2,2025
4,AI00005,AI Consultant,54624,EUR,EN,PT,France,S,Singapore,100,...,2025-06-23,1989,6.6,Advanced Robotics,15,4,2025,23,6,2025


In [11]:
data.drop(['job_id','posting_date','application_deadline'],axis=1,inplace=True)
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 22 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   job_title               15000 non-null  object 
 1   salary_usd              15000 non-null  int64  
 2   salary_currency         15000 non-null  object 
 3   experience_level        15000 non-null  object 
 4   employment_type         15000 non-null  object 
 5   company_location        15000 non-null  object 
 6   company_size            15000 non-null  object 
 7   employee_residence      15000 non-null  object 
 8   remote_ratio            15000 non-null  int64  
 9   required_skills         15000 non-null  object 
 10  education_required      15000 non-null  object 
 11  years_experience        15000 non-null  int64  
 12  industry                15000 non-null  object 
 13  job_description_length  15000 non-null  int64  
 14  benefits_score          15000 non-null

In [12]:
le = LabelEncoder()
cat = data.select_dtypes(include='object').columns
for i in cat:
    data[i] = le.fit_transform(data[i])

data.head(2)

,job_title,salary_usd,salary_currency,experience_level,employment_type,company_location,company_size,employee_residence,remote_ratio,required_skills,...,industry,job_description_length,benefits_score,company_name,posting_Date,posting_month,posting_year,application_date,application_month,application_year
0,3,90376,2,3,0,3,1,3,50,12938,...,0,1076,5.9,14,18,10,2024,7,11,2024
1,4,61895,2,0,0,2,1,9,100,1779,...,9,1268,5.2,15,20,11,2024,11,1,2025


In [13]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 22 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   job_title               15000 non-null  int32  
 1   salary_usd              15000 non-null  int64  
 2   salary_currency         15000 non-null  int32  
 3   experience_level        15000 non-null  int32  
 4   employment_type         15000 non-null  int32  
 5   company_location        15000 non-null  int32  
 6   company_size            15000 non-null  int32  
 7   employee_residence      15000 non-null  int32  
 8   remote_ratio            15000 non-null  int64  
 9   required_skills         15000 non-null  int32  
 10  education_required      15000 non-null  int32  
 11  years_experience        15000 non-null  int64  
 12  industry                15000 non-null  int32  
 13  job_description_length  15000 non-null  int64  
 14  benefits_score          15000 non-null

In [14]:
data['benefits_score'] = data['benefits_score'].astype('int')

In [15]:
X = data.drop(['salary_usd'],axis=1)
y = data['salary_usd']

In [16]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2)

In [17]:
data.head()

,job_title,salary_usd,salary_currency,experience_level,employment_type,company_location,company_size,employee_residence,remote_ratio,required_skills,...,industry,job_description_length,benefits_score,company_name,posting_Date,posting_month,posting_year,application_date,application_month,application_year
0,3,90376,2,3,0,3,1,3,50,12938,...,0,1076,5,14,18,10,2024,7,11,2024
1,4,61895,2,0,0,2,1,9,100,1779,...,9,1268,5,15,20,11,2024,11,1,2025
2,5,152626,2,2,1,17,0,15,0,5167,...,2,1974,9,3,18,3,2025,7,4,2025
3,16,80215,2,3,1,8,1,8,50,11651,...,1,1345,8,9,23,12,2024,24,2,2025
4,1,54624,0,0,3,6,2,14,100,6607,...,9,1989,6,1,15,4,2025,23,6,2025


In [18]:
sc = StandardScaler()
for i in X:
    data[[i]] = sc.fit_transform(data[[i]])

data.head()

,job_title,salary_usd,salary_currency,experience_level,employment_type,company_location,company_size,employee_residence,remote_ratio,required_skills,...,industry,job_description_length,benefits_score,company_name,posting_Date,posting_month,posting_year,application_date,application_month,application_year
0,-1.130279,90376,0.486844,1.341900,-1.347217,-1.116670,-0.000735,-1.120847,0.012660,1.541403,...,-1.624758,-0.741727,-1.429685,1.397916,0.258955,1.282527,-0.568933,-1.000193,1.578317,-0.708805
1,-0.957020,61895,0.486844,-1.347278,-1.347217,-1.290608,-0.000735,-0.080403,1.237809,-1.305243,...,0.454060,-0.408456,-1.429685,1.614218,0.486821,1.568364,-0.568933,-0.543223,-1.496646,1.410826
2,-0.783762,152626,0.486844,0.445507,-0.449431,1.318461,-1.225174,0.960041,-1.212489,-0.440968,...,-1.162798,0.817009,1.354874,-0.981405,0.258955,-0.718325,1.757675,-1.000193,-0.574157,1.410826
3,1.122078,80215,0.486844,1.341900,-0.449431,-0.246980,-0.000735,-0.253811,0.012660,1.213091,...,-1.393778,-0.274801,0.658734,0.316406,0.828620,1.854200,-0.568933,0.941929,-1.189150,1.410826
4,-1.476795,54624,-2.239555,-1.347278,1.346139,-0.594856,1.223704,0.786634,1.237809,-0.073626,...,0.454060,0.843046,-0.733546,-1.414009,-0.082845,-0.432489,1.757675,0.827687,0.040836,1.410826


In [19]:
models = {
    "Linear Regression": {
        "model": LinearRegression(),
        "params": {}  # No hyperparameters for LinearRegression by default
    },
    "Decision Tree": {
        "model": DecisionTreeRegressor(),
        "params": {
            'max_depth': [3, 5, 10],
            'min_samples_split': [2, 5, 10]
        }
    },
    "Random Forest": {
        "model": RandomForestRegressor(),
        "params": {
            'n_estimators': [50, 100],
            'max_depth': [3, 5, 10]
        }
    },
    "XGBoost": {
        "model": XGBRegressor(objective='reg:squarederror'),
        "params": {
            'n_estimators': [50, 100],
            'max_depth': [3, 5, 10],
            'learning_rate': [0.01, 0.1]
        }
    }
}


In [20]:
results = []

for name, m in models.items():
    print(f"\nRunning GridSearchCV for {name}")
    grid = GridSearchCV(m["model"], m["params"], cv=5, scoring='r2')
    grid.fit(X_train, y_train)
    
    best_model = grid.best_estimator_
    best_score = grid.best_score_
    best_name = name
    y_pred = best_model.predict(X_test)
    
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        'Model': name,
        'Best Params': grid.best_params_,
        'Model': name,
        'MSE': mse,
        'MAE': mae,
        'R2 Score': r2
    })

# Convert results to DataFrame
results_data = pd.DataFrame(results)
print(results_data)
print("===============================================================================")
print(f"\nBest Model: {best_name}")
print("===============================================================================")
print(f"Best R2 Score: {best_score:.2f}")



Running GridSearchCV for Linear Regression

Running GridSearchCV for Decision Tree

Running GridSearchCV for Random Forest

Running GridSearchCV for XGBoost
               Model                                        Best Params  \
0  Linear Regression                                                 {}   
1      Decision Tree         {'max_depth': 10, 'min_samples_split': 10}   
2      Random Forest             {'max_depth': 10, 'n_estimators': 100}   
3            XGBoost  {'learning_rate': 0.1, 'max_depth': 5, 'n_esti...   

            MSE           MAE  R2 Score  
0  1.356209e+09  26377.593520  0.622577  
1  5.498116e+08  15832.185410  0.846991  
2  4.137291e+08  14509.864005  0.884862  
3  3.959439e+08  14351.798085  0.889812  

Best Model: XGBoost
Best R2 Score: 0.88


In [21]:
feature_columns = X.columns.to_list()

In [22]:
joblib.dump(feature_columns,'features_ai_job.joblib')

['features_ai_job.joblib']

In [23]:
joblib.dump(best_model,'best_model_ai_job.joblib')

['best_model_ai_job.joblib']

In [ ]:
import streamlit as st
import pandas as pd
import joblib

model = joblib.load('best_model_ai_job.joblib')
features = joblib.load('features_ai_job.joblib')
st.title('AI JOB')

user_input={}
for i in features:
    user_input[i] = st.number_input(i,value=0.0)

if st.button('Predict'):
    data = pd.DataFrame([user_input])
    pred = model.predict(data)[0]
    st.success(f'Prediction : {(pred)}')

In [ ]:
cd .venv
streamlit run filename.py